# Tutorial 02: Quantização em FPGA (PTQ vs QAT com Brevitas)

Neste tutorial, exploramos a técnica mais importante de otimização em hardware para redes neurais: **a Quantização**.

### O que você vai aprender:
1. **Post-Training Quantization (PTQ)**: Reduzir os bits de uma rede pré-treinada (16 bits -> 8 bits -> 6 bits) diretamente no `hls4ml` e analisar a degradação de precisão.
2. **Quantization-Aware Training (QAT)**: Treinar a rede em PyTorch com a biblioteca oficial **Brevitas**, onde o modelo aprende a operar diretamente com **6 bits**.
3. **Exportação QONNX e Conversão no hls4ml**: Integrar o grafo quantizado e convertê-lo em hardware C++.
4. **Comparativo Final**: Avaliar curvas ROC e tabelas de acurácia mostrando como o QAT recupera a precisão de um modelo de 6 bits.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

sys.path.append('../..')
import models
import plotting
import hls4ml

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from brevitas.export import export_qonnx
import qonnx.util.cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.gemm_to_matmul import GemmToMatMul

print(f"Versão hls4ml: {hls4ml.__version__}")
print(f"Versão PyTorch: {torch.__version__}")

## 1. Carregando os Dados de Teste
Carregamos as matrizes pré-processadas do LHC Jet Tagging (16 variáveis, 5 classes).

In [ ]:
data_dir = '../../data/jet-tagging'
X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))
classes = np.load(os.path.join(data_dir, 'classes.npy'), allow_pickle=True)

eval_n = min(5000, len(X_test))
print(f"Avaliando {eval_n} amostras de teste.")

## 2. Modelo Baseline (Float32)
Carregamos a rede `JetTagger` treinada no Tutorial 01.

In [ ]:
model_baseline = models.JetTagger()
weights_path = '../../models/pytorch_weights_part1.pt'
model_baseline.load_state_dict(torch.load(weights_path))
model_baseline.eval()

with torch.no_grad():
    y_baseline_pred = model_baseline(torch.tensor(X_test[:eval_n], dtype=torch.float32)).numpy()

acc_baseline = accuracy_score(np.argmax(y_test[:eval_n], axis=1), np.argmax(y_baseline_pred, axis=1))
print(f"Acurácia Baseline PyTorch (Float32): {acc_baseline * 100:.2f}%")

## 3. Post-Training Quantization (PTQ Direto no hls4ml)
Vamos testar o que acontece quando simplesmente forçamos menos bits em hardware sem re-treinar o modelo:
- `ap_fixed<16,6>`: 16 bits
- `ap_fixed<8,3>`: 8 bits
- `ap_fixed<6,2>`: 6 bits

In [ ]:
ptq_results = {}

for prec in ['ap_fixed<16,6>', 'ap_fixed<8,3>', 'ap_fixed<6,2>']:
    cfg = hls4ml.utils.config_from_pytorch_model(
        model_baseline,
        input_shape=(16,),
        granularity='model',
        backend='Vivado',
    )
    cfg['Model']['Precision']['default'] = prec
    cfg['Model']['ReuseFactor'] = 1
    
    out_dir = f"hls_prj_ptq_{prec.replace('<','_').replace('>','').replace(',','_')}"
    h_model = hls4ml.converters.convert_from_pytorch_model(
        model_baseline,
        hls_config=cfg,
        output_dir=out_dir,
        part='xcu200-fsgd2104-2-e'
    )
    h_model.compile()
    y_ptq = h_model.predict(X_test[:eval_n])
    acc_ptq = accuracy_score(np.argmax(y_test[:eval_n], axis=1), np.argmax(y_ptq, axis=1))
    ptq_results[prec] = (acc_ptq, y_ptq)
    print(f"PTQ com {prec:18}: {acc_ptq * 100:.2f}% (Queda: {abs(acc_baseline - acc_ptq)*100:.2f}%)")

## 4. Quantization-Aware Training (QAT com Brevitas)
Agora instanciamos o `JetTaggerBrevitas`. Durante o treinamento, a biblioteca simula as restrições de ponto fixo de 6 bits nos pesos e ativações ReLU, permitindo que a rede compense a perda de precisão.

In [ ]:
model_brevitas = models.JetTaggerBrevitas()
brevitas_weights_path = '../../models/brevitas_weights_part2.pt'

if os.path.exists(brevitas_weights_path):
    print("Carregando pesos pré-treinados do Brevitas...")
    model_brevitas.load_state_dict(torch.load(brevitas_weights_path))
else:
    print("Treinando rede Brevitas por 5 épocas...")
    n_samples = 50000
    train_x = torch.tensor(X_train[:n_samples], dtype=torch.float32)
    train_y = torch.tensor(np.argmax(y_train[:n_samples], axis=1), dtype=torch.long)
    loader = DataLoader(TensorDataset(train_x, train_y), batch_size=256, shuffle=True)
    
    opt = torch.optim.Adam(model_brevitas.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model_brevitas.train()
    for ep in range(1, 6):
        tot_loss = 0.0
        for bx, by in loader:
            opt.zero_grad()
            crit(model_brevitas.logits(bx), by).backward()
            opt.step()
    torch.save(model_brevitas.state_dict(), brevitas_weights_path)
    print("Pesos QAT salvos!")

model_brevitas.eval()
with torch.no_grad():
    y_brevitas_float = model_brevitas(torch.tensor(X_test[:eval_n], dtype=torch.float32)).numpy()
acc_brevitas_float = accuracy_score(np.argmax(y_test[:eval_n], axis=1), np.argmax(y_brevitas_float, axis=1))
print(f"Acurácia Brevitas (6-bit em software): {acc_brevitas_float * 100:.2f}%")

## 5. Exportando para QONNX e Convertendo com hls4ml
Exportamos o modelo Brevitas para o formato aberto **QONNX** e o compilamos com o `hls4ml`.

In [ ]:
raw_onnx = 'brevitas_raw.onnx'
clean_onnx = 'brevitas_clean.onnx'

export_qonnx(model_brevitas, args=torch.randn(1, 16), export_path=raw_onnx, opset_version=14, dynamo=False)
qonnx.util.cleanup.cleanup(raw_onnx, out_file=clean_onnx)
m_wrap = ModelWrapper(clean_onnx)
m_wrap = m_wrap.transform(GemmToMatMul())
m_wrap = qonnx.util.cleanup.cleanup_model(m_wrap)
m_wrap.save(clean_onnx)

onnx_cfg = hls4ml.utils.config_from_onnx_model(m_wrap, granularity='name', backend='Vivado')
hls_brevitas = hls4ml.converters.convert_from_onnx_model(
    m_wrap,
    output_dir='hls4ml_prj_brevitas_6bit',
    backend='Vivado',
    hls_config=onnx_cfg,
    part='xcu200-fsgd2104-2-e'
)
hls_brevitas.compile()

y_hls_brevitas = hls_brevitas.predict(np.ascontiguousarray(X_test[:eval_n], dtype=np.float32))
acc_hls_brevitas = accuracy_score(np.argmax(y_test[:eval_n], axis=1), np.argmax(y_hls_brevitas, axis=1))
print(f"Acurácia hls4ml Brevitas 6-bit em Hardware: {acc_hls_brevitas * 100:.2f}%")

if os.path.exists(raw_onnx): os.remove(raw_onnx)
if os.path.exists(clean_onnx): os.remove(clean_onnx)

## 6. Comparação Visual e Curvas ROC
Plotamos o gráfico comparativo:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
plotting.makeRoc(y_test[:eval_n], y_baseline_pred, list(classes), linestyle='-')
plt.gca().set_prop_cycle(None)
plotting.makeRoc(y_test[:eval_n], ptq_results['ap_fixed<6,2>'][1], list(classes), linestyle=':')
plt.gca().set_prop_cycle(None)
plotting.makeRoc(y_test[:eval_n], y_hls_brevitas, list(classes), linestyle='--')

from matplotlib.lines import Line2D
lines = [
    Line2D([0], [0], color='black', linestyle='-', label='PyTorch Baseline (Float32)'),
    Line2D([0], [0], color='black', linestyle=':', label='PTQ Direto (ap_fixed<6,2>)'),
    Line2D([0], [0], color='black', linestyle='--', label='QAT Brevitas (6 bits)')
]
ax.legend(handles=lines, loc='lower right')
plt.title('Comparativo de Quantização: Float32 vs PTQ vs QAT')
plt.tight_layout()
plt.show()